# [LAB] LangChain 실습

LangChain으로 체인을 만들고, 도구를 연결하고, 답을 정해진 모양으로 받고, 대화를 기억하게 합니다. 보기 중 하나를 골라 주석을 풀어 코드를 완성하고, 완성된 코드는 실행하며 값을 바꿔 봅니다.

- **2장 LCEL과 Runnable**: 프롬프트·모델·파서를 `|`로 잇고, 같은 체인을 여러 방식으로 실행합니다.
- **3장 도구 호출**: 모델이 도구를 고르고, 코드가 실행하고, 결과를 다시 모델에 넣는 반복을 완성합니다.
- **4장 구조화된 출력**: 영화 리뷰를 정해진 모양으로 받는 두 방법을 비교합니다.
- **5장 세션 메모리**: 세션별 대화 기록을 체인에 연결합니다.
- **6장 상태 안에 담는 메모리**: 같은 일을 LangGraph로 해 보고 5장과 비교합니다. 고를 곳 없이 실행만 합니다.

셀은 위에서부터 차례로 실행합니다. 각 장 첫머리의 접힌 준비 코드도 실행해야 그 아래 셀이 동작합니다. 보기를 고르는 셀은 고른 뒤 실행하고 다음 셀로 넘어갑니다.

In [ ]:
%cd ../../../course/Chapter01/notebooks

**목차**

- 1. 실습 방법
- 2. LCEL과 Runnable
  - 2.1 기본 체인 만들기
  - 2.2 invoke·stream·batch로 실행하기
  - 2.3 RunnableParallel로 결과 모으기
  - 2.4 RunnablePassthrough로 값 덧붙이기
  - 2.5 RunnableLambda로 함수 끼우기
- 3. 도구 호출
  - 3.1 도구 정의 읽기
  - 3.2 도구 목록·연결·이름표 만들기
  - 3.3 도구 호출 루프 완성하기
  - 3.4 질문해 보기
- 4. 구조화된 출력
  - 4.1 출력 스키마 읽기
  - 4.2 방법 1: PydanticOutputParser
  - 4.3 방법 2: with_structured_output
- 5. 세션 메모리
  - 5.1 세션별 기록 저장소 만들기
  - 5.2 체인과 기록 연결하기
  - 5.3 세션 A·B로 대화해 보기
- 6. 상태 안에 담는 메모리
  - 6.1 State와 노드
  - 6.2 그래프 만들기
  - 6.3 스레드 A·B와 스냅샷
  - 6.4 AI에게 물어보기
- 7. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env()

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고친 뒤 바로 위 환경 점검 셀을 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

<details><summary>강사용 메모</summary>

- 원본: `src/raw/Chapter01/02-langchain/`의 5개 파일(6장만 솔루션). 빌드할 때 셀로 옮겨 넣으므로 수강생은 .py 파일을 보지 않습니다. 노트북에서만 손본 곳(patch)은 장마다 메모에 적었습니다.
- LLM 호출이 많습니다(전체 약 25회, 2장만 약 10회). 사내 게이트웨이의 tool calling(3장)·json_schema 응답 형식(4.3) 지원은 미확인입니다.
- CodeMate가 노트북에서 선택한 셀을 읽는지, 셀을 직접 고칠 수 있는지는 강의장에서 미확인입니다. 못 읽으면 셀 코드를 지시문 아래에 붙여 넣게 하고, 못 고치면 채팅 답의 코드를 빈칸에 붙여 넣게 합니다.
- 지시문 완성본은 주지 않고 프롬프트 힌트와 성공 기준만 줍니다. 성공 기준은 자동 검사하지 않습니다(2026-09-28 결정).

</details>

## 1. 실습 방법

코드 셀에서 손댈 곳은 주석으로 표시되어 있습니다.

- `# ✍️ CHANGE_HERE:` 바로 아래 줄의 값을 바꿔 다시 실행해 봅니다.
- `# ✅ CHOOSE_HERE:` 아래에 보기가 주석으로 들어 있습니다. 줄 끝의 번호(1️⃣~4️⃣)와 설명을 보고 하나를 고릅니다. 같은 번호가 붙은 줄을 모두 선택하고 `Ctrl + /`를 눌러 주석을 푼 뒤 실행합니다.

실행한 뒤에는 **성공 기준**과 비교해 봅니다. 🟢는 되어야 하는 것, ⛔는 다시 해야 하는 신호입니다. ⛔이면 푼 줄을 다시 선택하고 `Ctrl + /`로 주석 처리한 뒤 다른 보기를 고릅니다. 마지막으로 접힌 "정답 보기"에서 보기마다의 해설을 읽습니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. LCEL과 Runnable

프롬프트, 모델, 파서를 `|`로 이어 체인을 만들고, 같은 체인을 `invoke`·`stream`·`batch`로 실행합니다. 이어서 `RunnableParallel`·`RunnablePassthrough`·`RunnableLambda`로 결과를 모으고, 값을 덧붙이고, 함수를 체인에 끼워 봅니다.

2.1에서 고를 곳은 두 군데입니다. 두 곳을 모두 고르고 실행해야 2.2가 실행됩니다. 2.2부터는 완성된 코드를 실행하고, `# ✍️ CHANGE_HERE:` 줄의 값을 바꿔 다시 실행해 봅니다.

**왜 배우나:** 체인은 뒤의 RAG와 Agent 코드에서도 계속 쓰는 기본 모양입니다. 직접 한 번 이어 보면 AI가 만든 긴 체인도 `|` 순서대로 읽을 수 있습니다.

준비 코드입니다. 체인에 쓸 도구(프롬프트 템플릿, 출력 파서, Runnable 조합 도구)를 불러오고 `model = get_llm()`으로 모델을 준비합니다. 접혀 있으니 펼치지 않아도 됩니다. 실행만 합니다.

In [ ]:
"""
LCEL Runnable: LangChain 구성 요소를 파이프라인으로 연결하기

핵심 개념:
  - Runnable = invoke, stream, batch를 공통으로 제공하는 실행 단위
  - LCEL = prompt → model → parser 흐름을 | 연산자로 선언하는 방식
  - RunnableParallel, RunnablePassthrough, RunnableLambda로 입력 흐름을 조합 가능

"""

import os

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import (
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)

from config.llm import get_llm


load_dotenv()

model = get_llm()

<details><summary>강사용 메모</summary>

- 원본: `src/raw/Chapter01/02-langchain/01_lcel_runnable.py`. 원본 파일은 그대로 두고 노트북에서만 손봤습니다(patch).
- 뺀 것: 이전 강사 말투 주석(`# 가우스`, `# 사용자가 입력한 값 -> …`, `# 사용자마다 서로 다른 내용의 주제를…`, `# 사용자 입력 -> 모델에게 전달 -> …`, `# 파이썬 주석처리 -> … ctrl + /`, `# 주의) 변수명이 pros, cons로 고정인게 아님! 이거 내가 맘음대로 지을 수 있음!`), 주석 처리된 `runnable.stream({'country': 'USA'})` 예시(조각을 키별로 이어 붙여 `pprint`), `translate_acronym` 안의 죽은 코드 2줄(`# response = model.invoke()`), docstring의 실행 경로.
- 솔루션을 따른 것: 주석 처리된 `invoke`/`stream`/`batch` 실행 줄을 실행되게 했고, 프롬프트 문구를 `너는 초보자를 위한 AI 튜터야.` / `{topic}을 초급자에게 3문장으로 설명해줘.`로 맞췄습니다(뜻 차이 없음).
- 뺀 주석 중 "키 이름은 마음대로 지을 수 있다"는 2.3 설명으로 옮겼습니다.

</details>

### 2.1 기본 체인 만들기

**✅ CHOOSE_HERE: system, human 메시지 튜플을 작성해 주세요.**

`ChatPromptTemplate.from_messages([...])` 안에 메시지 두 개를 `(역할, 내용)` 튜플로 넣습니다. system 메시지에는 "초보자를 위한 AI 튜터"라는 역할을, human 메시지에는 `{topic}` 자리가 있는 질문을 담습니다. 중괄호 `{topic}`은 실행할 때 값이 들어갈 빈자리입니다.

보기 고르는 법: 같은 번호의 줄을 모두 선택하고 `Ctrl + /`를 누르면 주석이 풀립니다.

**성공 기준**

- 🟢 system, human 두 줄의 주석을 풀었고, 셀이 오류 없이 실행된다
- 🟢 2.2의 첫 답이 주제를 초보자 눈높이로 3문장 안팎으로 설명한다
- ⛔ `Unexpected message type` 오류가 난다 → 튜플이 `(역할, 내용)` 순서인 보기인지 확인합니다
- ⛔ 2.2에서 `missing variables` 오류가 나거나 모델이 주제를 되묻는다 → human 메시지에 중괄호로 쓴 `{topic}`이 있는 보기인지 확인합니다

**✅ CHOOSE_HERE: 위에 정의한 프롬프트, 모델, 출력 파서를 파이프라인 연산자로 연결해 주세요.**

`|`로 프롬프트 → 모델 → 파서를 차례로 잇습니다. 적은 순서가 곧 데이터가 흐르는 방향입니다.

보기 고르는 법: 같은 번호의 줄을 모두 선택하고 `Ctrl + /`를 누르면 주석이 풀립니다.

**성공 기준**

- 🟢 2.2의 첫 답이 `content=`가 붙은 묶음이 아니라 답변 텍스트만 출력된다
- ⛔ 답에 `content=`나 `AIMessage`가 보인다 → 체인 끝에 `parser`가 있는 보기인지 확인합니다
- ⛔ 2.2에서 `Invalid input type`이나 `ValidationError`가 난다 → `prompt` → `model` → `parser` 순서인 보기인지 확인합니다

In [ ]:
# ── 기본 LCEL 체인 ────────────────────────────────────────────────────────────
# ChatPromptTemplate: `{}` 변수만 바꿔 재사용할 수 있는 프롬프트 템플릿
prompt = ChatPromptTemplate.from_messages(
    [
        # ✅ CHOOSE_HERE: system, human 메시지 튜플을 작성해 주세요.
        # (같은 번호의 줄을 모두 선택하고 Ctrl + / 를 누르면 주석이 풀립니다)
        # ("system", "너는 초보자를 위한 AI 튜터야."),        # 1️⃣ human 자리 이름이 subject
        # ("human", "{subject}을 초급자에게 3문장으로 설명해줘."),  # 1️⃣
        #
        # ("system", "너는 초보자를 위한 AI 튜터야."),        # 2️⃣ 중괄호 없이 topic을 씀
        # ("human", "topic을 초급자에게 3문장으로 설명해줘."),  # 2️⃣
        #
        # ("system", "너는 초보자를 위한 AI 튜터야."),        # 3️⃣ human 자리 이름이 topic
        # ("human", "{topic}을 초급자에게 3문장으로 설명해줘."),  # 3️⃣
        #
        # ("너는 초보자를 위한 AI 튜터야.", "system"),        # 4️⃣ (내용, 역할) 순서로 씀
        # ("{topic}을 초급자에게 3문장으로 설명해줘.", "human"),  # 4️⃣
    ]
)

# StrOutputParser: LLM 출력을 문자열로 변환
parser = StrOutputParser()

# LCEL: LangChain Expression Language
# - 프롬프트 작성 → LLM 호출 → 출력 파싱 과정을 하나의 실행 단위로 묶는다.
# - prompt, model, parser는 모두 Runnable 계열 객체다.
# - Runnable | Runnable 결과도 Runnable이므로 체인을 계속 이어붙일 수 있다.
# ✅ CHOOSE_HERE: 위에 정의한 프롬프트, 모델, 출력 파서를 파이프라인 연산자로 연결해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + / 를 누르면 주석이 풀립니다)
# chain = prompt | model           # 1️⃣ 프롬프트 → 모델
# chain = model | prompt | parser  # 2️⃣ 모델 → 프롬프트 → 파서
# chain = prompt | parser | model  # 3️⃣ 프롬프트 → 파서 → 모델
# chain = prompt | model | parser  # 4️⃣ 프롬프트 → 모델 → 파서

<details><summary>정답 보기</summary>

**system, human 메시지 튜플을 작성해 주세요.**

**정답: 3️⃣**

- 1️⃣ human 메시지의 자리 이름이 `{subject}`라서 2.2에서 넣은 `topic`과 맞지 않습니다. 2.2에서 `KeyError: Input to ChatPromptTemplate is missing variables {'subject'}`가 납니다.
- 2️⃣ 중괄호가 없어 `topic`이 값이 들어갈 자리가 아니라 그냥 글자가 됩니다. 오류 없이 2.2에서 모델이 "설명할 주제를 알려 주세요"처럼 주제를 되묻거나 'topic'이라는 낱말을 설명합니다.
- 3️⃣ `(역할, 내용)` 순서의 튜플 두 개입니다. human 메시지의 `{topic}` 자리에 2.2에서 넣은 주제가 들어갑니다.
- 4️⃣ 튜플 순서가 `(내용, 역할)`로 뒤바뀌어 내용을 역할 이름으로 읽습니다. 이 셀에서 바로 `ValueError: Unexpected message type`이 납니다.

**위에 정의한 프롬프트, 모델, 출력 파서를 파이프라인 연산자로 연결해 주세요.**

**정답: 4️⃣**

- 1️⃣ 파서가 빠져 답이 `AIMessage` 묶음으로 나옵니다. 2.2 첫 답에 `content=…`가 붙어 출력되고, 마지막 `batch` 줄에서 `AttributeError: 'AIMessage' object has no attribute 'strip'`이 납니다.
- 2️⃣ 모델이 맨 앞이라 `{"topic": …}` dict를 모델이 먼저 받습니다. 2.2에서 `ValueError: Invalid input type <class 'dict'>`가 납니다.
- 3️⃣ 파서가 모델보다 앞에 있어 모델의 답이 아니라 프롬프트를 글자로 바꾸려다 실패합니다. 2.2에서 `ValidationError … Input should be a valid string`이 납니다.
- 4️⃣ 프롬프트가 만든 메시지를 모델에 넣고, 모델의 답을 파서가 글자만 남깁니다. 적은 순서대로 데이터가 흐릅니다.

</details>

### 2.2 invoke·stream·batch로 실행하기

**✍️ CHANGE_HERE: 주제를 바꿔 봅니다**

먼저 그대로 실행해 세 가지 실행 결과를 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 `"LLM"`을 다른 주제(예: `"벡터 데이터베이스"`)로 바꿔 다시 실행합니다. 아래 코드 셀은 모델을 네 번 호출하므로 조금 기다립니다.

2.4 이후에 아래 코드 셀을 다시 실행하려면 2.1 셀부터 다시 실행합니다(2.4·2.5가 `chain`을 바꿉니다).

In [ ]:
# Runnable 실행 방식:
# - invoke(): 단일 입력 처리
# - stream(): 토큰/청크 단위 출력
# - batch(): 여러 입력 일괄 처리

# ✍️ CHANGE_HERE: 주제를 바꿔 봅니다
answer = chain.invoke({"topic": "LLM"})
print(answer)
print("----")
for chunk in chain.stream({"topic": "LangChain"}):
    print(chunk, end="", flush=True)
print()

print("----")
inputs = [{"topic": "LangChain Expression Language"}, {"topic": "LangChain Runnable"}]
responses = chain.batch(inputs)
for topic_input, response in zip(inputs, responses):
    print(f"{topic_input.get('topic')}: {response.strip()}")

**성공 기준**

- 🟢 `----` 선을 사이에 두고 답이 세 덩어리로 출력된다: 한 번에 받은 답, 조각으로 이어 나온 답, 두 주제의 답
- 🟢 첫 답이 바꾼 주제를 설명한다

### 2.3 RunnableParallel로 결과 모으기

**✍️ CHANGE_HERE: 나라를 바꿔 봅니다**

**실행 전에 읽습니다.** 이 셀은 모델을 네 번 호출하고 출력이 깁니다. `pros`와 `cons` 아래 목록의 첫 줄만 봐도 됩니다.

`# ✍️ CHANGE_HERE:` 바로 아래 줄의 `"베트남"`을 다른 나라로 바꿔 다시 실행합니다. 아래쪽 dict 방식은 베트남 그대로 두고 모양만 비교합니다. `RunnableParallel(pros=…, cons=…)`의 키 이름 `pros`, `cons`는 정해진 이름이 아니라 마음대로 지을 수 있습니다.

In [ ]:
# ── RunnableParallel ──────────────────────────────────────────────────────────
# 여러 Runnable을 병렬로 실행해 결과를 dict로 모을 때 사용
pros_chain = (
    ChatPromptTemplate.from_template(
        "{country}로 여행갈 때의 장점을, 5개 정도 목록 형식으로 설명해줘."
    )
    | model
    | StrOutputParser()
)
cons_chain = (
    ChatPromptTemplate.from_template(
        "{country}로 여행갈 때의 단점을, 5개 정도 목록 형식으로 설명해줘."
    )
    | model
    | StrOutputParser()
)

runnable = RunnableParallel(pros=pros_chain, cons=cons_chain)
# ✍️ CHANGE_HERE: 나라를 바꿔 봅니다
answers = runnable.invoke({"country": "베트남"})
print("pros")
print(answers.get("pros"))

print("cons")
print(answers.get("cons"))


# Python dict가 pipe 사이에 들어가면 자동으로 RunnableParallel처럼 동작한다.
parallel_dict = RunnablePassthrough() | {
    "pros": ChatPromptTemplate.from_template(
        "{country}로 여행갈 때의 장점을, 5개 정도 목록 형식으로 설명해줘."
    )
    | model
    | StrOutputParser(),
    "cons": ChatPromptTemplate.from_template(
        "{country}로 여행갈 때의 단점을, 5개 정도 목록 형식으로 설명해줘."
    )
    | model
    | StrOutputParser(),
}

answers = parallel_dict.invoke({"country": "베트남"})
print("pros")
print(answers.get("pros"))

print("cons")
print(answers.get("cons"))

**성공 기준**

- 🟢 `pros`와 `cons` 아래에 장점 목록과 단점 목록이 각각 출력된다
- 🟢 위쪽은 바꾼 나라의 장단점이고, 아래쪽 dict 방식도 같은 모양(pros·cons 두 목록)으로 출력된다

### 2.4 RunnablePassthrough로 값 덧붙이기

**✍️ CHANGE_HERE: 질문을 바꿔 봅니다**

`# ✍️ CHANGE_HERE:` 바로 아래 줄의 질문을 더 길거나 짧게 바꿔 실행합니다. 입력 dict에 `length`가 덧붙는지 봅니다.

In [ ]:
# ── RunnablePassthrough ───────────────────────────────────────────────────────
# 입력값을 그대로 다음 단계에 전달하면서 부가 필드를 추가할 때 사용
def get_question_length(text: str):
    return len(text)


# 예: 사용자 질문을 바탕으로 맥락 정보나 메타데이터를 추가
passthrough = RunnablePassthrough.assign(
    length=lambda x: get_question_length(x["question"])
)
# ✍️ CHANGE_HERE: 질문을 바꿔 봅니다
print(passthrough.invoke({"question": "RunnablePassthrough의 역할이 뭐야?"}))

prompt = ChatPromptTemplate.from_template("{question} 3문장으로 설명해줘.")
chain = passthrough | prompt | model | StrOutputParser()

from draw import draw

draw(chain)  # 체인 구조를 그림으로 봅니다
print(chain.invoke({"question": "RunnablePassthrough의 역할이 뭐야?"}))

**성공 기준**

- 🟢 첫 줄에 원래 `question`과 새 `length`(질문 글자 수)가 함께 든 dict가 출력된다
- 🟢 질문을 바꾸면 `length` 숫자도 바뀐다

### 2.5 RunnableLambda로 함수 끼우기

**✍️ CHANGE_HERE: 약어를 바꿔 봅니다**

`# ✍️ CHANGE_HERE:` 바로 아래 줄의 `"GPT"`를 사전에 있는 `"LLM"`이나 사전에 없는 약어(예: `"RAG"`)로 바꿔 실행합니다.

In [ ]:
# ── RunnableLambda ────────────────────────────────────────────────────────────
# 일반 Python 함수를 Runnable로 감싸 체인 안에서 사용할 때 활용
acronym_dict = {
    "GPT": "Generative Pre-trained Transformer",
    "LLM": "Large Language Model",
    "LCEL": "LangChain Expression Language",
}


def translate_acronym(acronym: str):
    return acronym_dict.get(acronym, acronym)


# ✍️ CHANGE_HERE: 약어를 바꿔 봅니다
print(RunnableLambda(translate_acronym).invoke("GPT"))

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 초보자를 위한 AI 튜터야."),
        ("human", "{topic}을 초급자에게 3문장으로 설명해줘."),
    ]
)

chain = (
    {
        "topic": RunnableLambda(translate_acronym),
    }
    | prompt
    | model
    | StrOutputParser()
)
print(chain.invoke("GPT"))

**성공 기준**

- 🟢 사전에 있는 약어는 풀어 쓴 이름으로, 사전에 없는 약어는 그대로 출력된다
- 🟢 마지막 답이 `GPT`를 풀어 쓴 이름(Generative Pre-trained Transformer)으로 설명한다

**볼 지점**
- 2.2: 같은 `chain`을 호출 방법만 바꿔 실행했습니다. `invoke`는 답을 한 번에, `stream`은 조각으로 이어서, `batch`는 두 주제의 답을 목록으로 돌려줍니다.
- 2.3: `RunnableParallel`은 두 체인의 결과를 `pros`·`cons` 키가 있는 dict 하나로 모읍니다. dict를 `|` 사이에 두어도 같은 일을 합니다.
- 2.4: `RunnablePassthrough.assign`은 입력을 그대로 넘기면서 `length` 값을 덧붙입니다.
- 2.5: `RunnableLambda`로 감싼 함수는 체인 안에서 한 단계가 됩니다. 마지막 체인은 약어를 풀어 쓴 이름을 `{topic}` 자리에 넣습니다.

<details><summary>강사용 메모</summary>

- 2.3 dict 방식의 `RunnablePassthrough() | {...}`는 dict 앞에 Runnable 하나를 두어 dict가 RunnableParallel로 바뀌게 한 모양입니다.
- 셀을 건너뛰면 2.4·2.5에서 `prompt`·`chain` 이름이 다시 정의되므로, 2.2를 다시 실행하려면 2.1부터 다시 실행합니다.

</details>

## 3. 도구 호출

모델에게 날씨 조회 도구와 계산 도구를 알려 주고, 질문에 맞는 도구를 모델이 고르게 합니다. 모델이 고른 도구는 코드가 실행하고, 결과를 다시 모델에게 넣어 최종 답을 받습니다.

3.1은 도구를 만드는 코드를 읽고 실행만 합니다. `@tool`을 붙인 함수의 docstring(함수 첫 줄의 설명 글)은 모델에게 주는 도구 설명이 되고, 타입 힌트(`city: str`)는 인자 타입이 됩니다. 고를 곳은 3.2에 세 군데, 3.3에 한 군데입니다. 3.4는 완성된 코드를 실행하고 도시를 바꿔 봅니다.

**왜 배우나:** Agent가 도구를 쓰는 방식이 바로 이 반복입니다. 등록 → 선택 → 실행 → 결과 전달을 한 번 직접 이어 보면, 도구가 늘어나도 같은 구조로 읽을 수 있습니다.

준비 코드입니다. 메시지 종류와 도구를 만드는 `@tool`을 불러옵니다. 펼치지 않고 실행만 해도 됩니다.

In [ ]:
"""
Tool Calling: LLM이 외부 도구를 자율적으로 선택·실행하기

핵심 흐름:
  사용자 질문 → LLM이 필요한 도구 결정 → 도구 실행 → 결과 반환

"""

import os

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool

from config.llm import get_llm

load_dotenv();

<details><summary>강사용 메모</summary>

- 원본: `src/raw/Chapter01/02-langchain/02_tool_calling.py`. 원본 파일은 그대로 두고 노트북에서만 손봤습니다(patch).
- 뺀 것: 이전 강사 주석(description은 LLM을 위한 설명, 타입 힌트도 LLM에게 힌트, "그건 어떻게하는데?"), `bind_tools` 주석 2줄("시퀀스를 전달받을 수 있음", "한번에 여러 도구들"), `tool_map` 위의 `'''{'get_weahter': …}'''` 문자열(오타 포함), 경로 주석 `05-langgraph/03_tool_node.py 참고`(2026-09-28 사용자 결정), docstring의 실행 경로. "docstring·타입 힌트가 모델에게 주는 힌트"라는 내용은 장 소개 글로 옮겼습니다.
- `tool_map`은 솔루션 표기 `{t.name: t for t in tools}`를 씁니다(`tool` 이름이 데코레이터 `@tool`과 겹치지 않게).
- `print(call)`은 raw에만 있는 줄이지만 호출 정보(name·args·id)를 화면에 보여 주려고 남겼습니다. 솔루션에는 없습니다.

</details>

### 3.1 도구 정의 읽기

In [ ]:
# ── 도구 정의 (@tool 데코레이터) ────────────────────────────────────────────────
# 함수의 docstring이 LLM에게 전달되는 도구 설명이 된다.
# 타입 힌트가 JSON Schema로 변환되어 파라미터 형식을 LLM에게 알려준다.


@tool
def get_weather(city: str) -> str:
    """지정한 도시의 현재 날씨를 조회한다."""
    fake_data = {
        "서울": "맑음, 기온 22°C",
        "부산": "흐림, 기온 24°C",
        "제주": "비, 기온 20°C",
    }
    return fake_data.get(city, f"{city}의 날씨 데이터를 찾을 수 없습니다.")


@tool
def calculate(expression: str) -> str:
    """수학 수식을 계산한다. 예: '2 + 3 * 4'"""
    try:
        result = eval(expression, {"__builtins__": {}})  # noqa: S307
        return str(result)
    except Exception as e:
        return f"계산 오류: {e}"

### 3.2 도구 목록·연결·이름표 만들기

**✅ CHOOSE_HERE: LLM에 연결할 도구 목록을 작성해 주세요.**

3.1에서 `@tool`로 만든 두 도구를 목록 하나에 담습니다. 이 목록이 모델에게 알려 줄 도구 명단입니다.

보기 고르는 법: 같은 번호의 줄을 모두 선택하고 `Ctrl + /`를 누르면 주석이 풀립니다.

**성공 기준**

- 🟢 셀이 오류 없이 실행된다
- 🟢 3.4에서 날씨 질문에는 `get_weather`, 계산 질문에는 `calculate` 호출 정보 줄이 나온다
- ⛔ 이 셀에서 `TypeError`나 `Unsupported function` 오류가 난다 → 도구를 괄호나 따옴표 없이 담은 보기인지 확인합니다
- ⛔ 3.4의 계산 질문에 호출 정보 줄이 없다 → 두 도구가 모두 든 보기인지 확인합니다

**✅ CHOOSE_HERE: bind_tools()를 사용해 LLM에 도구 스키마를 연결해 주세요.**

`bind_tools()`는 모델을 호출할 때마다 도구 명세(이름·설명·인자)를 함께 보내게 합니다. 결과는 새 이름 `llm_with_tools`에 담습니다. 3.3의 루프가 이 이름을 씁니다.

보기 고르는 법: 같은 번호의 줄을 모두 선택하고 `Ctrl + /`를 누르면 주석이 풀립니다.

**성공 기준**

- 🟢 3.4의 날씨 질문에서 호출 정보 줄이 최종 답보다 먼저 나온다
- ⛔ 3.4에서 `NameError: name 'llm_with_tools'`가 난다 → 결과를 `llm_with_tools`에 담는 보기인지 확인합니다
- ⛔ 호출 정보 줄 없이 날씨를 확인할 수 없다고 답한다 → `bind_tools(tools)`로 도구를 연결한 보기인지 확인합니다

**✅ CHOOSE_HERE: 도구 이름으로 실제 도구를 찾을 수 있도록 매핑을 만들어 주세요.**

모델은 쓸 도구를 이름(문자열)으로만 알려 줍니다. 그래서 이름을 넣으면 실제 도구가 나오는 dict(`{이름: 도구}`)를 미리 만들어 둡니다.

보기 고르는 법: 같은 번호의 줄을 모두 선택하고 `Ctrl + /`를 누르면 주석이 풀립니다.

**성공 기준**

- 🟢 세 곳을 모두 고른 뒤 셀이 오류 없이 실행된다
- 🟢 3.4의 날씨 답이 도구 결과(예: 서울은 맑음, 기온 22°C)를 쓴다
- ⛔ 이 셀이나 3.4에서 `KeyError`나 `TypeError`가 난다 → 도구 이름을 키로, 도구를 값으로 담는 dict 보기인지 확인합니다

In [ ]:
# ✅ CHOOSE_HERE: LLM에 연결할 도구 목록을 작성해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + / 를 누르면 주석이 풀립니다)
# tools = [get_weather(), calculate()]  # 1️⃣ 두 도구를 불러 그 결과를 담음
# tools = [get_weather, calculate]      # 2️⃣ 두 도구를 목록에 담음
# tools = ["get_weather", "calculate"]  # 3️⃣ 두 도구의 이름을 글자로 담음
# tools = [get_weather]                 # 4️⃣ 날씨 도구만 담음

# ── LLM에 도구 바인딩 ─────────────────────────────────────────────────────────
# bind_tools()는 LLM 호출 시마다 도구 스키마를 자동으로 포함시킨다.
llm = get_llm()
# ✅ CHOOSE_HERE: bind_tools()를 사용해 LLM에 도구 스키마를 연결해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + / 를 누르면 주석이 풀립니다)
# llm_with_tools = llm.bind_tools(tools)    # 1️⃣ 도구를 연결한 모델을 새 이름에 담음
# llm_with_tools = llm                      # 2️⃣ 원래 모델을 새 이름에 담음
# llm.bind_tools(tools)                     # 3️⃣ 도구 연결만 하고 담지 않음
# llm_with_tools = llm.bind_tools([tools])  # 4️⃣ 도구 목록을 한 번 더 감싸 연결

# 도구 이름 → 실행 함수 매핑
# ✅ CHOOSE_HERE: 도구 이름으로 실제 도구를 찾을 수 있도록 매핑을 만들어 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + / 를 누르면 주석이 풀립니다)
# tool_map = {t: t.name for t in tools}                   # 1️⃣ 도구를 키로, 이름을 값으로
# tool_map = {"weather": get_weather, "calc": calculate}  # 2️⃣ 이름을 손으로 적은 dict
# tool_map = [t.name for t in tools]                      # 3️⃣ 도구 이름 목록
# tool_map = {t.name: t for t in tools}                   # 4️⃣ 이름을 키로, 도구를 값으로

<details><summary>정답 보기</summary>

**LLM에 연결할 도구 목록을 작성해 주세요.**

**정답: 2️⃣**

- 1️⃣ `@tool`을 붙인 함수는 이미 도구 객체라 괄호를 붙여 부를 수 없습니다. 이 셀에서 `TypeError: 'StructuredTool' object is not callable`이 납니다.
- 2️⃣ 3.1에서 만든 두 도구를 그대로 목록에 담습니다. 이 목록이 모델에게 알려 줄 도구 명단이 됩니다.
- 3️⃣ 도구가 아니라 이름 글자만 담았습니다. 바로 아래 `bind_tools` 줄에서 `ValueError: Unsupported function`이 납니다.
- 4️⃣ 계산 도구가 빠졌습니다. 오류 없이 3.4의 계산 질문에 호출 정보 줄이 나오지 않고 모델이 혼자 계산해 답합니다. 답이 맞아 보여도 도구를 쓰지 않은 것입니다.

**bind_tools()를 사용해 LLM에 도구 스키마를 연결해 주세요.**

**정답: 1️⃣**

- 1️⃣ 도구 명세를 붙인 새 모델을 `llm_with_tools`에 담습니다. 3.3의 루프가 이 이름으로 모델을 부릅니다.
- 2️⃣ 도구를 연결하지 않은 모델을 그대로 씁니다. 오류 없이 3.4의 날씨 질문에 호출 정보 줄이 나오지 않고 "실시간 날씨를 확인할 수 없다"고 답합니다.
- 3️⃣ `bind_tools()`는 `llm`을 바꾸지 않고 새 모델을 돌려줍니다. 돌려받은 것을 담지 않아 3.4에서 `NameError: name 'llm_with_tools' is not defined`가 납니다.
- 4️⃣ 목록 안에 목록을 넣어 도구 하나로 읽힙니다. 이 셀에서 `ValueError: Unsupported function`이 납니다.

**도구 이름으로 실제 도구를 찾을 수 있도록 매핑을 만들어 주세요.**

**정답: 4️⃣**

- 1️⃣ 키와 값이 뒤바뀌어 도구 객체를 키로 씁니다. 이 셀에서 `TypeError: unhashable type: 'StructuredTool'`이 납니다.
- 2️⃣ 키를 `weather`, `calc`로 손으로 적었습니다. 모델은 `get_weather`라는 이름으로 부르므로 3.4에서 `KeyError: 'get_weather'`가 납니다.
- 3️⃣ dict가 아니라 이름 목록을 만듭니다. 3.4에서 이름으로 찾다가 `TypeError: list indices must be integers or slices, not str`가 납니다.
- 4️⃣ 도구마다 `.name`(예: `get_weather`)을 키로, 도구 자체를 값으로 담습니다. 모델이 알려 준 이름으로 실제 도구를 바로 찾습니다.

</details>

### 3.3 도구 호출 루프 완성하기

**✅ CHOOSE_HERE: tool_calls의 이름과 인자를 사용해 해당 도구를 실행해 주세요.**

`response.tool_calls`의 항목 `call` 하나에는 도구 이름(`name`), 인자(`args`), 호출 번호(`id`)가 들어 있습니다. 이름으로 `tool_map`에서 도구를 찾고, 인자를 넣어 실행한 결과를 `result`에 담습니다. 바로 아래 줄이 `result`를 `ToolMessage`로 만들어 대화에 되돌려 넣습니다.

보기 고르는 법: 같은 번호의 줄을 모두 선택하고 `Ctrl + /`를 누르면 주석이 풀립니다.

**성공 기준**

- 🟢 셀이 오류 없이 실행되고 아무것도 출력되지 않는다(함수만 만듭니다)
- 🟢 3.4에서 도구 호출 정보 줄(`{'name': 'get_weather', 'args': {'city': '서울'}, …}`)이 출력된 뒤 최종 답이 나온다
- ⛔ 3.4에서 `NameError: name 'result'`가 난다 → 실행 결과를 `result`에 담는 보기인지 확인합니다
- ⛔ 3.4에서 `unhashable type: 'dict'`가 난다 → 도구를 `call["name"]`으로 찾는 보기인지 확인합니다
- ⛔ 호출 정보 줄은 나왔는데 날씨를 가져오지 못했다고 답한다 → 도구에 `call["args"]`를 넣는 보기인지 확인합니다

In [ ]:
# ── 수동 도구 호출 루프 ───────────────────────────────────────────────────────
# LLM 응답에 tool_calls가 있으면 실행 후 결과를 다시 LLM에 전달한다.
# LangGraph의 ToolNode가 이 루프를 자동화한다.


def run_with_tools(question: str) -> str:
    messages = [HumanMessage(content=question)]

    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            return response.content

        for call in response.tool_calls:
            print(call)
            # ✅ CHOOSE_HERE: tool_calls의 이름과 인자를 사용해 해당 도구를 실행해 주세요.
            # (같은 번호의 줄을 모두 선택하고 Ctrl + / 를 누르면 주석이 풀립니다)
            # tool_fn = tool_map[call["name"]]       # 1️⃣ 이름으로 찾고 인자를 넣어 실행
            # result = tool_fn.invoke(call["args"])  # 1️⃣
            #
            # tool_fn = tool_map[call["args"]]       # 2️⃣ 인자로 찾고 인자를 넣어 실행
            # result = tool_fn.invoke(call["args"])  # 2️⃣
            #
            # tool_fn = tool_map[call["name"]]       # 3️⃣ 이름으로 찾고 이름을 넣어 실행
            # result = tool_fn.invoke(call["name"])  # 3️⃣
            #
            # tool_fn = tool_map[call["name"]]       # 4️⃣ 실행 결과를 output에 담음
            # output = tool_fn.invoke(call["args"])  # 4️⃣
            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=call["id"],
                )
            )

<details><summary>정답 보기</summary>

**정답: 1️⃣**

- 1️⃣ 이름으로 도구를 찾고, 모델이 채운 인자(`args`)를 넣어 실행합니다. 결과가 `result`에 담겨 바로 아래 `ToolMessage`로 대화에 들어갑니다.
- 2️⃣ 도구를 이름이 아니라 인자 dict로 찾습니다. 3.4에서 `TypeError: unhashable type: 'dict'`가 납니다.
- 3️⃣ 인자 대신 도구 이름 `"get_weather"`를 넣어 실행합니다. 도구는 그 글자를 도시 이름으로 읽습니다. 오류 없이 3.4의 날씨 답이 "날씨 정보를 가져오지 못했다"고 나옵니다.
- 4️⃣ 결과를 `output`에 담아 아래 줄이 찾는 `result`가 없습니다. 3.4에서 `NameError: name 'result' is not defined`가 납니다.

</details>

### 3.4 질문해 보기

**✍️ CHANGE_HERE: 도시를 바꿔 봅니다**

처음에는 고치지 않고 실행합니다. 질문 세 개를 차례로 보내고, 질문마다 모델을 두 번 이상 호출하므로 조금 기다립니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 `"서울"`을 `"제주"`나 가짜 날씨 데이터에 없는 도시(예: `"대전"`)로 바꿔 다시 실행합니다.

In [ ]:
# ── 실행 예시 ─────────────────────────────────────────────────────────────────

print("=== 날씨 질문 ===")
# ✍️ CHANGE_HERE: 도시를 바꿔 봅니다
print(run_with_tools("서울 날씨 알려줘"))

print("\n=== 수학 계산 ===")
print(run_with_tools("(123 + 456) * 2를 계산해줘"))

print("\n=== 복합 질문 ===")
print(run_with_tools("서울과 부산 날씨를 비교해줘"))

# LangGraph ToolNode 방식으로 전환하면 이 루프가 필요 없어진다.

**성공 기준**

- 🟢 질문마다 도구 호출 정보 줄(도구 이름과 인자)이 먼저 출력되고, 그 뒤에 최종 답이 출력된다
- 🟢 날씨 질문에는 `get_weather`, 계산 질문에는 `calculate`가 호출되었다
- 🟢 데이터에 없는 도시로 바꾸면 "날씨 데이터를 찾을 수 없습니다"라는 도구 결과를 바탕으로 답한다

**볼 지점**
- 호출 정보 줄의 `name`은 모델이 고른 도구, `args`는 모델이 채운 인자, `id`는 호출 번호입니다. 모델은 도구를 실행하지 않고 이 정보만 돌려줍니다. 도구 이름과 인자를 `tool`·`arguments`라고 적는 자료도 있습니다. 이름만 다르고 담긴 정보는 같습니다.
- 도구를 실행한 것은 `run_with_tools()` 안의 코드입니다. 실행 결과는 `ToolMessage`로 대화에 들어가고, 모델이 그것을 읽고 최종 답을 만듭니다.
- 복합 질문(서울과 부산)에서는 호출 정보 줄이 두 번 나올 수 있습니다. 모델이 도구를 두 번 고른 것입니다.

<details><summary>강사용 메모</summary>

- 사내 게이트웨이에서 tool calling(`bind_tools`)이 동작하는지 미확인입니다. 안 되면 성공 기준과 정답 보기만 읽고 넘어갑니다.
- 슬라이드 JSON(`tool`/`arguments`)과 호출 정보(`name`/`args`/`id`)의 이름이 다릅니다. 슬라이드를 가리키며 대응을 짚어 줍니다.
- `calculate`는 `eval`을 씁니다(builtins 비움). 수업용 가짜 도구입니다. 물으면 "실서비스에서는 쓰지 않는다"고만 답합니다.
- LangGraph의 `ToolNode`가 이 루프를 자동화합니다(2일차 `src/raw/Chapter01/04-langgraph/03_tool_node.py`).

</details>

## 4. 구조화된 출력

영화 리뷰를 글이 아니라 `MovieReview`라는 정해진 모양(제목·장르·평점·요약·추천 여부)으로 받습니다. 두 가지 방법을 씁니다.

- 방법 1 `PydanticOutputParser`(4.2, 보기를 고릅니다): 프롬프트에 형식 지시를 넣고, 모델의 글 답을 파서가 읽어 `MovieReview`로 바꿉니다.
- 방법 2 `with_structured_output()`(4.3, 완성된 코드): 모델에게 스키마를 직접 넘겨 형식 지시 없이 `MovieReview`를 받습니다.

4.1의 스키마는 읽고 실행만 합니다. 필드마다 붙은 `description`이 모델에게 전달되는 설명입니다.

**왜 배우나:** 답이 정해진 모양으로 와야 코드가 평점으로 거르거나, 저장하거나, 화면에 표로 보여 줄 수 있습니다.

준비 코드입니다. 파서, 프롬프트 템플릿, 스키마를 만드는 Pydantic 도구를 불러오고 모델을 준비합니다. 내용은 몰라도 되니 실행하고 넘어갑니다.

In [ ]:
"""
Output Parser: LLM 출력을 구조화된 데이터로 변환하기

두 가지 방식 비교:
  1. PydanticOutputParser + format_instructions → 프롬프트에 형식 지시 포함
  2. .with_structured_output()                 → LLM의 function-calling 기능 활용 (권장)

"""

from dotenv import load_dotenv
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

from config.llm import get_llm

load_dotenv()

llm = get_llm()

<details><summary>강사용 메모</summary>

- 원본: `src/raw/Chapter01/02-langchain/03_output_parser.py`. 원본 파일은 그대로 두고 노트북에서만 손봤습니다(patch).
- 방법 2는 raw에서 전부 주석(TODO 2곳 포함)이라 솔루션 코드로 채웠습니다. PDF에 없는 개념이라 예제로 둡니다(2026-09-28 사용자 결정).
- 솔루션을 따른 것: system 프롬프트를 `…응답해.\n{format_instructions}`로(형식 지시가 줄을 바꿔 들어감), human 프롬프트를 `{movie} 리뷰 작성해줘`로.
- 뺀 것: `# review1은 MovieReview의 인스턴스가 되길 희망한다.` 주석, 끝의 ASCII 비교 표(장 끝 "볼 지점"의 표로 옮김), docstring의 실행 경로.

</details>

### 4.1 출력 스키마 읽기

In [ ]:
# ── 출력 스키마 정의 ───────────────────────────────────────────────────────────


class MovieReview(BaseModel):
    title: str = Field(description="영화 제목")
    genre: str = Field(description="장르 (예: 액션, 드라마, SF)")
    rating: float = Field(description="평점 (1.0 ~ 10.0)")
    summary: str = Field(description="100자 이내 한 줄 요약")
    recommend: bool = Field(description="추천 여부")

### 4.2 방법 1: PydanticOutputParser

**✅ CHOOSE_HERE: MovieReview 스키마를 사용하는 PydanticOutputParser를 생성해 주세요.**

`PydanticOutputParser`에 4.1의 `MovieReview`를 알려 주면, 모델의 글 답을 `MovieReview`로 바꾸는 파서가 됩니다.

4.2의 코드 셀은 고를 곳이 세 군데입니다. 세 곳을 모두 고른 뒤 한 번 실행합니다.

보기 고르는 법: 같은 번호의 줄을 모두 선택하고 `Ctrl + /`를 누르면 주석이 풀립니다.

**성공 기준**

- 🟢 `parser = …` 보기의 주석을 풀었다
- ⛔ `TypeError`나 `ValidationError`가 난다 → `pydantic_object=`에 `MovieReview`를 괄호 없이 넘기는 보기인지 확인합니다
- ⛔ `NameError: name 'MovieReview'`가 난다 → 4.1 셀을 먼저 실행했는지 확인합니다

**✅ CHOOSE_HERE: format_instructions가 포함된 프롬프트를 구성해 주세요.**

system 메시지에 `{format_instructions}` 자리를 두고, `.partial()`로 그 자리에 파서가 만든 형식 지시를 미리 채워 둡니다. human 메시지에는 `{movie}` 자리가 있습니다.

보기 고르는 법: 같은 번호의 줄을 모두 선택하고 `Ctrl + /`를 누르면 주석이 풀립니다.

**성공 기준**

- 🟢 셀 아래에 제목·장르·평점·요약·추천 다섯 줄이 모두 출력된다
- ⛔ `OutputParserException`이 난다 → system 메시지에 `{format_instructions}` 자리가 있는 보기인지 확인합니다
- ⛔ `missing variables` 오류가 난다 → `.partial()`이 있고 human 메시지의 자리 이름이 `{movie}`인 보기인지 확인합니다

**✅ CHOOSE_HERE: 프롬프트, LLM, 파서를 LCEL 체인으로 연결해 주세요.**

2장에서 만든 체인과 같은 모양입니다. 이번에는 마지막 단계가 문자열 파서가 아니라 `MovieReview` 파서입니다.

보기 고르는 법: 같은 번호의 줄을 모두 선택하고 `Ctrl + /`를 누르면 주석이 풀립니다.

**성공 기준**

- 🟢 평점이 숫자로, 추천이 ✓ 또는 ✗로 출력된다
- ⛔ `has no attribute 'title'`이나 `'genre'` 오류가 난다 → 체인 끝이 `parser`인 보기인지 확인합니다
- ⛔ `NameError: name 'chain1'`이 난다 → 세 번째 고를 곳까지 주석을 풀었는지 확인합니다

In [ ]:
# ── 방법 1: PydanticOutputParser ──────────────────────────────────────────────
# 프롬프트에 JSON 형식 지시문을 주입하여 LLM이 구조화된 출력을 생성하도록 유도.
# LLM이 지시를 따르지 않으면 파싱 실패 가능.

print("=== 방법 1: PydanticOutputParser ===")

# ✅ CHOOSE_HERE: MovieReview 스키마를 사용하는 PydanticOutputParser를 생성해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + / 를 누르면 주석이 풀립니다)
# parser = PydanticOutputParser(MovieReview)                  # 1️⃣ 스키마를 이름 없이 넘김
# parser = PydanticOutputParser(pydantic_object=MovieReview())  # 2️⃣ MovieReview()를 만들어 넘김
# parser = PydanticOutputParser(pydantic_object=MovieReview)  # 3️⃣ 스키마 클래스를 이름 붙여 넘김
# parser = PydanticOutputParser                               # 4️⃣ 괄호 없이 PydanticOutputParser를 담음

# ✅ CHOOSE_HERE: format_instructions가 포함된 프롬프트를 구성해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + / 를 누르면 주석이 풀립니다)
# prompt = ChatPromptTemplate.from_messages(                  # 1️⃣ 형식 지시 자리만 두고 채우지 않음
#     [                                                       # 1️⃣
#         ("system", "영화 리뷰 전문가야. 다음 형식으로 응답해.\n{format_instructions}"),  # 1️⃣
#         ("human", "{movie} 리뷰 작성해줘"),                 # 1️⃣
#     ]                                                       # 1️⃣
# )                                                           # 1️⃣
#
# prompt = ChatPromptTemplate.from_messages(                  # 2️⃣ 형식 지시 자리를 두고 .partial()로 채움
#     [                                                       # 2️⃣
#         ("system", "영화 리뷰 전문가야. 다음 형식으로 응답해.\n{format_instructions}"),  # 2️⃣
#         ("human", "{movie} 리뷰 작성해줘"),                 # 2️⃣
#     ]                                                       # 2️⃣
# ).partial(format_instructions=parser.get_format_instructions())  # 2️⃣
#
# prompt = ChatPromptTemplate.from_messages(                  # 3️⃣ 형식 지시 자리 없이 .partial()만 씀
#     [                                                       # 3️⃣
#         ("system", "영화 리뷰 전문가야. 다음 형식으로 응답해."),  # 3️⃣
#         ("human", "{movie} 리뷰 작성해줘"),                 # 3️⃣
#     ]                                                       # 3️⃣
# ).partial(format_instructions=parser.get_format_instructions())  # 3️⃣
#
# prompt = ChatPromptTemplate.from_messages(                  # 4️⃣ human 자리 이름이 title
#     [                                                       # 4️⃣
#         ("system", "영화 리뷰 전문가야. 다음 형식으로 응답해.\n{format_instructions}"),  # 4️⃣
#         ("human", "{title} 리뷰 작성해줘"),                 # 4️⃣
#     ]                                                       # 4️⃣
# ).partial(format_instructions=parser.get_format_instructions())  # 4️⃣

# ✅ CHOOSE_HERE: 프롬프트, LLM, 파서를 LCEL 체인으로 연결해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + / 를 누르면 주석이 풀립니다)
# chain1 = prompt | llm                      # 1️⃣ 프롬프트 → 모델
# chain1 = prompt | llm | parser             # 2️⃣ 프롬프트 → 모델 → MovieReview 파서
# chain1 = prompt | parser | llm             # 3️⃣ 프롬프트 → MovieReview 파서 → 모델
# chain1 = prompt | llm | StrOutputParser()  # 4️⃣ 프롬프트 → 모델 → 문자열 파서
review1: MovieReview = chain1.invoke({"movie": "인터스텔라"})

print(f"제목   : {review1.title}")
print(f"장르   : {review1.genre}")
print(f"평점   : {review1.rating}")
print(f"요약   : {review1.summary}")
print(f"추천   : {'✓' if review1.recommend else '✗'}")

<details><summary>정답 보기</summary>

**MovieReview 스키마를 사용하는 PydanticOutputParser를 생성해 주세요.**

**정답: 3️⃣**

- 1️⃣ `PydanticOutputParser`에는 스키마를 `pydantic_object=`라는 이름을 붙여 넘겨야 합니다. 이 셀에서 `TypeError: BaseModel.__init__() takes 1 positional argument but 2 were given`가 납니다.
- 2️⃣ 괄호를 붙여 값이 하나도 없는 `MovieReview`를 만들려다 실패합니다. 이 셀에서 `ValidationError: 5 validation errors for MovieReview`(Field required)가 납니다.
- 3️⃣ 스키마인 `MovieReview` 클래스 자체를 알려 주어, 모델의 글 답을 `MovieReview`로 바꾸는 파서를 만듭니다.
- 4️⃣ 괄호가 없어 파서를 만들지 않고 파서의 종류만 담습니다. 두 번째 고를 곳의 `parser.get_format_instructions()`에서 `TypeError: … missing 1 required positional argument: 'self'`가 납니다.

**format_instructions가 포함된 프롬프트를 구성해 주세요.**

**정답: 2️⃣**

- 1️⃣ `{format_instructions}` 자리는 있지만 `.partial()`로 채우지 않았습니다. `chain1.invoke` 줄에서 `KeyError: … missing variables {'format_instructions'}`가 납니다.
- 2️⃣ system 메시지의 `{format_instructions}` 자리에 파서가 만든 형식 지시를 미리 넣고, `{movie}` 자리에는 실행할 때 값을 넣습니다.
- 3️⃣ 형식 지시를 만들었지만 넣을 자리가 프롬프트에 없어 모델에게 전달되지 않습니다. 모델이 형식 없이 글로 리뷰를 써서 `OutputParserException: Invalid json output`이 납니다.
- 4️⃣ human 메시지의 자리 이름이 `{title}`이라 넣는 값 `{"movie": …}`와 맞지 않습니다. `KeyError: … missing variables {'title'}`가 납니다.

**프롬프트, LLM, 파서를 LCEL 체인으로 연결해 주세요.**

**정답: 2️⃣**

- 1️⃣ 파서가 빠져 결과가 `AIMessage`입니다. `review1.title` 줄에서 `AttributeError: 'AIMessage' object has no attribute 'title'`이 납니다.
- 2️⃣ 2장과 같은 모양이고, 마지막 단계가 `MovieReview` 파서라서 결과를 `review1.title`처럼 필드 이름으로 꺼낼 수 있습니다.
- 3️⃣ 파서가 모델보다 앞에 있어 모델의 답이 아니라 프롬프트를 읽으려다 실패합니다. `ValidationError … Input should be a valid string`이 납니다.
- 4️⃣ 답을 글자로만 바꿉니다. 제목 줄에 `<built-in method title …>`이 찍히고, 장르 줄에서 `AttributeError: … has no attribute 'genre'`가 납니다. 2장 준비 코드를 실행하지 않았다면 `NameError: name 'StrOutputParser'`가 납니다.

</details>

### 4.3 방법 2: with_structured_output

**✍️ CHANGE_HERE: 영화를 바꿔 봅니다**

먼저 그대로 실행해 방법 1과 같은 다섯 줄이 나오는지 봅니다. 이 방법은 프롬프트에 형식 지시가 없습니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 `"기생충"`을 다른 영화로 바꿔 다시 실행합니다.

In [ ]:
# ── 방법 2: .with_structured_output() ────────────────────────────────────────
# LLM의 function-calling(tool use) 기능을 활용하므로 더 안정적.
# 프롬프트에 형식 지시를 추가할 필요 없음.

print("\n=== 방법 2: with_structured_output() ===")

structured_llm = llm.with_structured_output(MovieReview)

prompt2 = ChatPromptTemplate.from_messages(
    [
        ("system", "영화 리뷰 전문가야."),
        ("human", "{movie} 리뷰 작성해줘"),
    ]
)

chain2 = prompt2 | structured_llm
# ✍️ CHANGE_HERE: 영화를 바꿔 봅니다
review2: MovieReview = chain2.invoke({"movie": "기생충"})

print(f"제목   : {review2.title}")
print(f"장르   : {review2.genre}")
print(f"평점   : {review2.rating}")
print(f"요약   : {review2.summary}")
print(f"추천   : {'✓' if review2.recommend else '✗'}")

**성공 기준**

- 🟢 방법 1과 같은 다섯 줄(제목·장르·평점·요약·추천)이 형식 지시 없이 출력된다
- 🟢 바꾼 영화의 제목과 장르가 나온다

**볼 지점**
- 두 방법 모두 결과가 `MovieReview`라서 `review1.title`처럼 필드 이름으로 값을 꺼냅니다.
- 방법 1은 프롬프트의 형식 지시에 기대고, 방법 2는 프롬프트에 형식 지시가 없습니다.

두 방법을 비교하면 다음과 같습니다.

| | PydanticOutputParser | with_structured_output() |
|---|---|---|
| 안정성 | LLM 지시 이행에 의존 | function-calling 보장 |
| 프롬프트 복잡도 | format_instructions 필요 | 프롬프트 단순 |
| 구형 모델 지원 | ✓ (모든 텍스트 LLM) | function-calling 필요 |
| 권장 | 구형 모델 / 커스텀 파싱 | GPT-4o 이상 기본 권장 |

<details><summary>강사용 메모</summary>

- 비교 표는 원본 끝의 ASCII 표(`print('''┌───…''')`)를 그대로 옮겼습니다. ASCII 그림은 셀에서 뺐습니다(2026-09-28 사용자 결정).
- 표와 원본 주석의 "function-calling 보장", "GPT-4o 이상 기본 권장"은 원본 문구입니다. 현재 langchain-openai(1.6.6)의 `with_structured_output` 기본 방식은 `json_schema`(응답 형식 지정)이지 function calling이 아닙니다. 물으면 "모델이 스키마에 맞춰 답하도록 API 차원에서 지정한다"고 답합니다.
- 사내 게이트웨이의 json_schema 응답 형식 지원은 미확인입니다. 방법 2가 실패하면 `llm.with_structured_output(MovieReview, method="function_calling")`으로 바꿔 볼 수 있습니다(노트북 코드는 원본대로 둠).

</details>

## 5. 세션 메모리

대화 기록을 체인 밖의 저장소에 세션(대화방)별로 모아 두고, 요청할 때마다 그 기록을 프롬프트에 넣어 줍니다. 대화를 그대로 보관해 다시 넣는 이 방식을 Buffer Memory라고 부릅니다.

5.1과 5.2에서 한 군데씩 고릅니다. 5.3은 완성된 코드를 실행하고 이름을 바꿔 봅니다.

**왜 배우나:** 모델은 앞 대화를 기억하지 않습니다. 이 장의 `RunnableWithMessageHistory`가 "이전 대화를 다시 넣어 보내기"를 대신 해 줍니다.

준비 코드입니다. 대화 기록을 담는 도구와 `RunnableWithMessageHistory`를 불러오고 모델을 준비합니다. 접혀 있으니 실행만 합니다.

In [ ]:
"""
메모리 전략 1 — LangChain 방식: RunnableWithMessageHistory

대화 기록을 체인 외부(InMemoryChatMessageHistory)에서 세션별로 관리.
체인 자체는 순수 함수처럼 동작하고, 히스토리는 wrapper가 주입한다.

"""

from dotenv import load_dotenv
from langchain_core.chat_history import (
    BaseChatMessageHistory,
    InMemoryChatMessageHistory,
)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core._api import LangChainDeprecationWarning
import warnings

# 수업 화면 정리: 이 장에서 쓰는 두 도구의 사용 중단 예정 안내 경고만 숨깁니다.
warnings.filterwarnings(
    "ignore",
    message="RunnableWithMessageHistory is deprecated",
    category=LangChainDeprecationWarning,
)
warnings.filterwarnings(
    "ignore",
    message="The class `InMemoryChatMessageHistory` was deprecated",
    category=LangChainDeprecationWarning,
)

from config.llm import get_llm

load_dotenv()

llm = get_llm()

<details><summary>강사용 메모</summary>

- 원본: `src/raw/Chapter01/02-langchain/04_memory_runnable.py`. 원본 파일은 그대로 두고 노트북에서만 손봤습니다(patch).
- 뺀 것: `get_session_history` 안의 강사 말투 주석 4줄, 대화 누적 모양을 적은 `'''# 최초의 ChatPromptTemplate …'''` 문자열(5.2 설명의 text 블록으로 옮김), docstring의 실행 경로.
- 마지막 출력의 `LangGraph 방식(05_memory_langgraph.py)`을 `LangGraph 방식`으로 줄였습니다(2026-09-28 사용자 결정).
- 접힌 준비 코드에 경고 숨김 줄을 더했습니다. langchain-core 1.6.4에서 `RunnableWithMessageHistory`(5.2, "RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.")와 `InMemoryChatMessageHistory`(5.3 첫 호출, "The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 …")가 사용 중단 예정 경고를 냅니다. 사용자 결정(2026-09-28)으로 이 두 경고만(`LangChainDeprecationWarning` + 메시지 일치) 숨겼고 다른 경고는 그대로 보입니다. 결정 당시에는 앞의 것만 알려졌고, 뒤의 것은 실행 검증에서 발견해 같은 이유로 함께 숨겼습니다. 물으면 "LangChain이 LangGraph 방식을 권장한다는 안내"라고 답하고 6장으로 잇습니다.

</details>

### 5.1 세션별 기록 저장소 만들기

**✅ CHOOSE_HERE: session_id별 InMemoryChatMessageHistory를 생성하고 반환해 주세요.**

`get_session_history(session_id)`는 세션 번호를 받아 그 세션의 대화 기록을 돌려주는 함수입니다. 처음 보는 세션이면 빈 기록(`InMemoryChatMessageHistory()`)을 새로 만들어 `session_store`에 넣고, 이미 있으면 있던 기록을 돌려줍니다.

보기 고르는 법: 같은 번호의 줄을 모두 선택하고 `Ctrl + /`를 누르면 주석이 풀립니다.

**성공 기준**

- 🟢 셀이 오류 없이 실행되고 아무것도 출력되지 않는다
- 🟢 5.3에서 세션 A의 두 번째 답이 앞에서 말한 이름을 기억한다
- ⛔ 세션 A의 두 번째 답도 이름을 모른다 → 처음 보는 세션일 때만 새 기록을 만드는 보기인지 확인합니다
- ⛔ 5.3에서 `KeyError: 'session_A'`가 난다 → `if` 조건이 `not in`인 보기인지 확인합니다

In [ ]:
# ── 세션 저장소 ───────────────────────────────────────────────────────────────
# 실제 서비스에서는 Redis·DB로 교체 가능

session_store: dict[str, BaseChatMessageHistory] = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    # ✅ CHOOSE_HERE: session_id별 InMemoryChatMessageHistory를 생성하고 반환해 주세요.
    # (같은 번호의 줄을 모두 선택하고 Ctrl + / 를 누르면 주석이 풀립니다)
    # if session_id not in session_store:                     # 1️⃣ 처음 보는 세션만 새 기록을 만들어 돌려줌
    #     session_store[session_id] = InMemoryChatMessageHistory()  # 1️⃣
    # return session_store[session_id]                        # 1️⃣
    #
    # return InMemoryChatMessageHistory()                     # 2️⃣ 부를 때마다 새 기록을 돌려줌
    #
    # session_store[session_id] = InMemoryChatMessageHistory()  # 3️⃣ 부를 때마다 새 기록을 넣고 돌려줌
    # return session_store[session_id]                        # 3️⃣
    #
    # if session_id in session_store:                         # 4️⃣ 이미 있는 세션만 새 기록을 만들어 돌려줌
    #     session_store[session_id] = InMemoryChatMessageHistory()  # 4️⃣
    # return session_store[session_id]                        # 4️⃣
    pass  # 그대로 둡니다

<details><summary>정답 보기</summary>

**정답: 1️⃣**

- 1️⃣ 처음 보는 세션일 때만 새 기록을 만들어 `session_store`에 넣고, 그 세션의 기록을 돌려줍니다. 같은 세션이면 이전 기록이 이어집니다.
- 2️⃣ 부를 때마다 빈 기록을 새로 만들어 돌려주고 저장하지도 않습니다. 오류 없이 5.3에서 세션 A의 두 번째 답이 이름을 모르고, "세션 A 저장된 메시지" 아래가 비어 있습니다.
- 3️⃣ 조건 없이 부를 때마다 그 세션의 기록을 빈 기록으로 덮어씁니다. 오류 없이 5.3에서 세션 A의 두 번째 답이 이름을 모르고, 저장된 메시지 아래도 비어 있습니다.
- 4️⃣ 조건이 반대라 처음 보는 세션에 기록을 만들지 않고 꺼내려 합니다. 5.3의 첫 호출에서 `KeyError: 'session_A'`가 납니다.

</details>

### 5.2 체인과 기록 연결하기

**✅ CHOOSE_HERE: RunnableWithMessageHistory로 체인과 세션 히스토리를 연결해 주세요.**

`RunnableWithMessageHistory`는 체인을 감싸서, 호출 전에 세션 기록을 프롬프트의 `history` 자리에 넣고 호출 뒤에 새 대화를 기록에 더합니다. 사용자 입력이 든 키 이름(`input`)과 기록을 넣을 자리 이름(`history`)을 알려 줘야 합니다.

같은 세션에서 두 번 물으면 요청이 이렇게 쌓입니다.

```text
첫 번째 요청: system, human(첫 번째 질문)
첫 번째 응답: ai(첫 번째 응답) → 기록에 저장
두 번째 요청: system, human(첫 번째 질문), ai(첫 번째 응답), human(두 번째 질문)
```

보기 고르는 법: 같은 번호의 줄을 모두 선택하고 `Ctrl + /`를 누르면 주석이 풀립니다.

**성공 기준**

- 🟢 셀이 오류 없이 실행되고 아무것도 출력되지 않는다
- 🟢 5.3에서 세션 A·B의 답이 오류 없이 출력된다
- ⛔ 이 셀에서 `TypeError`가 난다 → `get_session_history`를 괄호 없이 넘기는 보기인지 확인합니다
- ⛔ 5.3에서 `missing variables {'history'}`가 나거나 `KeyError('question')` 줄이 보인다 → 입력 키가 `input`, 기록 키가 `history`인 보기인지 확인합니다

In [ ]:
# ── 체인 구성 ─────────────────────────────────────────────────────────────────
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 친절한 AI 어시스턴트야."),
        MessagesPlaceholder(variable_name="history"),  # 이전 대화가 여기 삽입됨
        ("human", "{input}"),
    ]
)

chain = prompt | llm

# ✅ CHOOSE_HERE: RunnableWithMessageHistory로 체인과 세션 히스토리를 연결해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + / 를 누르면 주석이 풀립니다)
# chain_with_history = RunnableWithMessageHistory(  # 1️⃣ 기록 함수를 불러 그 결과를 넘김
#     chain,                                        # 1️⃣
#     get_session_history(),                        # 1️⃣
#     input_messages_key="input",                   # 1️⃣
#     history_messages_key="history",               # 1️⃣
# )                                                 # 1️⃣
#
# chain_with_history = RunnableWithMessageHistory(  # 2️⃣ 입력 키와 기록 키를 바꿔 씀
#     chain,                                        # 2️⃣
#     get_session_history,                          # 2️⃣
#     input_messages_key="history",                 # 2️⃣
#     history_messages_key="input",                 # 2️⃣
# )                                                 # 2️⃣
#
# chain_with_history = RunnableWithMessageHistory(  # 3️⃣ 입력 키 input, 기록 키 history
#     chain,                                        # 3️⃣
#     get_session_history,                          # 3️⃣
#     input_messages_key="input",                   # 3️⃣
#     history_messages_key="history",               # 3️⃣
# )                                                 # 3️⃣
#
# chain_with_history = RunnableWithMessageHistory(  # 4️⃣ 입력 키 question, 기록 키 history
#     chain,                                        # 4️⃣
#     get_session_history,                          # 4️⃣
#     input_messages_key="question",                # 4️⃣
#     history_messages_key="history",               # 4️⃣
# )                                                 # 4️⃣

<details><summary>정답 보기</summary>

**정답: 3️⃣**

- 1️⃣ 괄호를 붙여 함수를 그 자리에서 부릅니다. 이 셀에서 `TypeError: get_session_history() missing 1 required positional argument: 'session_id'`가 납니다. 함수 자체를 넘겨야 세션마다 불러 씁니다.
- 2️⃣ 두 키 이름이 뒤바뀌어 기록이 `history` 자리에 들어가지 않습니다. 5.3에서 `KeyError: … missing variables {'history'}`가 납니다.
- 3️⃣ 체인과 기록 함수를 넘기고, 입력 키 `input`과 기록 자리 `history`를 프롬프트의 이름과 맞춥니다.
- 4️⃣ 입력 키를 `question`이라 적었는데 넣는 dict에는 `input`만 있습니다. 답은 나오지만 대화가 기록에 저장되지 않아 세션 A의 두 번째 답이 이름을 모릅니다. 셀 아래에 `Error in RootListenersTracer.on_chain_end callback: KeyError('question')` 줄이 함께 보입니다.

</details>

### 5.3 세션 A·B로 대화해 보기

**✍️ CHANGE_HERE: 이름을 바꿔 봅니다**

먼저 그대로 실행합니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 이름을 바꿔 다시 실행합니다. 다시 실행하면 같은 세션에 대화가 더 쌓여 저장된 메시지 줄이 늘어납니다. 처음 상태로 보려면 5.1부터 다시 실행합니다.

In [ ]:
# ── 세션 A ────────────────────────────────────────────────────────────────────
print("=== 세션 A: 홍길동 ===")
config_a = {"configurable": {"session_id": "session_A"}}

# ✍️ CHANGE_HERE: 이름을 바꿔 봅니다
r1 = chain_with_history.invoke({"input": "내 이름은 홍길동이야."}, config=config_a)
print(f"AI: {r1.content}")

r2 = chain_with_history.invoke({"input": "내 이름이 뭐야?"}, config=config_a)
print(f"AI: {r2.content}")


# ── 세션 B ────────────────────────────────────────────────────────────────────
print("\n=== 세션 B: 별개 대화 ===")
config_b = {"configurable": {"session_id": "session_B"}}

r3 = chain_with_history.invoke({"input": "내 이름이 뭐야?"}, config=config_b)
print(f"AI: {r3.content}")  # B는 A 대화를 모름


# ── 저장된 히스토리 확인 ──────────────────────────────────────────────────────
print("\n=== 세션 A 저장된 메시지 ===")
for msg in get_session_history("session_A").messages:
    role = "사람" if msg.type == "human" else " AI "
    print(f"[{role}] {msg.content}")


# ── LangChain 방식의 한계 ─────────────────────────────────────────────────────
print("""
[LangChain 방식 한계]
- 히스토리가 체인 외부에 있어 그래프와 통합이 어려움
- 도구 실행 결과가 메시지 흐름에 자동 포함되지 않음
- 복잡한 분기·반복 구조에서 히스토리 관리가 번거로움

→ LangGraph 방식은 State 안에 messages를 두어 이 문제를 해결.
""")

**성공 기준**

- 🟢 세션 A의 두 번째 답에 바꾼 이름이 나온다
- 🟢 세션 B의 답은 이름을 모른다고 한다
- 🟢 "세션 A 저장된 메시지" 아래에 [사람]과 [ AI ] 줄이 번갈아 출력된다

**볼 지점**
- 세션 A는 같은 `session_id`로 두 번 물었기 때문에 두 번째 요청에 첫 대화가 함께 들어갔습니다.
- 세션 B는 `session_id`가 달라 기록이 비어 있습니다. 모델이 같아도 기록이 다르면 다른 대화가 됩니다.
- 기록은 체인 밖의 `session_store`에 있습니다. 마지막 출력은 이 방식의 한계를 적고, 다음 장의 LangGraph 방식으로 넘어갑니다.

## 6. 상태 안에 담는 메모리

같은 일을 LangGraph로 합니다. 대화 기록을 그래프의 State(그래프가 들고 다니는 정보 묶음) 안 `messages`에 두고, `MemorySaver`(스레드별 State를 저장해 두는 장치)가 스레드(대화방)별로 저장합니다. 5장과 이 장은 모두 대화를 그대로 쌓아 다시 넣는 Buffer 방식입니다. 다른 점은 기록을 두는 곳입니다.

LangGraph는 2일차에 자세히 배우므로 이 장은 고를 곳이 없습니다. 완성된 코드를 실행하고 5장과 비교합니다. 6.1의 `chatbot` 노드는 system 메시지와 State의 `messages`를 함께 모델에 보내고 답을 `messages`에 더합니다. 6.2는 그 노드 하나를 시작(START)과 끝(END) 사이에 두고 `MemorySaver`와 함께 그래프를 만듭니다.

**왜 배우나:** Agent를 만들 때는 대화 기록뿐 아니라 도구 결과, 분기 같은 흐름도 함께 관리해야 합니다. State 안에 기록을 두면 이것들을 한곳에서 다룰 수 있습니다.

준비 코드입니다. LangGraph 도구와 메시지 종류를 불러오고 모델을 준비합니다. 이 셀도 실행을 잊지 않습니다.

In [ ]:
"""
메모리 전략 2 — LangGraph 방식: State에 messages 내장

대화 기록이 State의 일부 → MemorySaver가 스레드별로 영속화.
외부 히스토리 관리 불필요. 도구 결과도 자동으로 메시지 흐름에 포함.

"""

from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

<details><summary>강사용 메모</summary>

- 원본: `src/solution/Chapter01/02-langchain/05_memory_langgraph.py`(raw는 TODO 3곳이 빈 상태: `return {'messages': []}`, `checkpointer = None`, `app = None`). LangGraph는 2-1에서 배우므로 예제로 실행만 합니다(목차 결정). 노트북에서만 손본 것: 끝의 ASCII 비교 표를 빼고 장 끝 "볼 지점"의 표로 옮김, docstring의 실행 경로 삭제.
- 같은 커널에서 6.3을 다시 실행하면 스레드 A 기록이 계속 쌓입니다. 처음 상태로 보려면 6.2부터 다시 실행합니다(`MemorySaver`를 새로 만듦).

</details>

### 6.1 State와 노드

In [ ]:
# ── State 정의 ────────────────────────────────────────────────────────────────
# add_messages reducer: 기존 메시지 리스트에 새 메시지를 누적(append) 한다.
# 덮어쓰기 방지 → 대화 히스토리가 자동으로 쌓인다.


class State(TypedDict):
    messages: Annotated[list, add_messages]


def chatbot(state: State) -> dict:
    system = SystemMessage(content="너는 친절한 AI 어시스턴트야.")
    return {"messages": [llm.invoke([system] + state["messages"])]}

### 6.2 그래프 만들기

In [ ]:
# ── 그래프 빌드 ───────────────────────────────────────────────────────────────
builder = StateGraph(State)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

# MemorySaver: 스레드별 State를 메모리에 유지 (프로세스 재시작 시 초기화)
checkpointer = MemorySaver()
app = builder.compile(checkpointer=checkpointer)

### 6.3 스레드 A·B와 스냅샷

**✍️ CHANGE_HERE: 스레드 이름을 바꿔 봅니다**

우선 코드 그대로 실행해 결과를 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 `"thread_B"`를 `"thread_A"`로 바꿔 이 셀을 다시 실행합니다. 스레드 A의 대화가 한 번 더 쌓이므로 스냅샷의 메시지 줄이 늘어나는 것은 정상입니다.

In [ ]:
# ── 스레드 A: 홍길동 ───────────────────────────────────────────────────────────
print("=== 스레드 A ===")
config_a = {"configurable": {"thread_id": "thread_A"}}

app.invoke(
    {"messages": [HumanMessage(content="내 이름은 홍길동이야.")]}, config=config_a
)
result = app.invoke(
    {"messages": [HumanMessage(content="내 이름이 뭐야?")]}, config=config_a
)
print(result["messages"][-1].content)


# ── 스레드 B: 별개 세션 ───────────────────────────────────────────────────────
print("\n=== 스레드 B (A와 독립) ===")
# ✍️ CHANGE_HERE: 스레드 이름을 바꿔 봅니다
config_b = {"configurable": {"thread_id": "thread_B"}}

result_b = app.invoke(
    {"messages": [HumanMessage(content="내 이름이 뭐야?")]}, config=config_b
)
print(result_b["messages"][-1].content)  # B는 A 대화를 모름


# ── 스냅샷 확인 ───────────────────────────────────────────────────────────────
print("\n=== 스레드 A 현재 State 스냅샷 ===")
snapshot = app.get_state(config_a)
for msg in snapshot.values["messages"]:
    role = "사람" if msg.type == "human" else " AI "
    print(f"[{role}] {msg.content}")

print(f"\n다음 실행 노드: {snapshot.next}")  # () = 완료 상태

**성공 기준**

- 🟢 바꾸기 전: 스레드 A는 이름(홍길동)을 기억하고, 스레드 B는 모른다
- 🟢 `thread_A`로 바꾸면 "스레드 B" 아래 답도 홍길동을 안다
- 🟢 스냅샷 아래에 [사람]과 [ AI ] 줄이 출력되고, 마지막 줄이 `다음 실행 노드: ()`다

**볼 지점**
- `add_messages`는 새 메시지를 덮어쓰지 않고 `messages` 뒤에 붙입니다. 그래서 대화가 State 안에 쌓입니다.
- 같은 `thread_id`로 호출하면 `MemorySaver`가 저장해 둔 State를 이어 씁니다. 그래서 스레드 A는 이름을 기억합니다.
- `app.get_state(config_a)`로 스레드 A의 State를 꺼내 저장된 대화를 볼 수 있습니다. `다음 실행 노드: ()`는 그래프 실행이 끝났다는 뜻입니다.

5장 방식과 비교하면 다음과 같습니다.

| | LangChain (RunnableWithMessageHistory) | LangGraph (MemorySaver) |
|---|---|---|
| 히스토리 위치 | 체인 외부 (session_store) | State 내부 (messages 필드) |
| 도구 결과 통합 | 수동 처리 필요 | 자동 (ToolMessage로 축적) |
| 분기/반복 | 구현 복잡 | 그래프 구조로 자연스럽게 표현 |
| 영속성 | 별도 저장소 연결 필요 | checkpointer 교체만으로 전환 |
| 권장 시나리오 | 단순 Q&A 챗봇 | Agent, 복잡한 대화 흐름 |

<details><summary>강사용 메모</summary>

- 비교 표는 원본 끝의 ASCII 표를 그대로 옮겼습니다(ASCII 그림은 셀에서 뺌, 2026-09-28 사용자 결정). 원본 표의 첫 열 이름 `RunnableWithHist`는 줄임말이라 전체 이름으로 적었습니다.

</details>

### 6.4 AI에게 물어보기

**프롬프트 힌트** 5장 세션 메모리의 셀들과 이 장의 셀들을 붙여 넣고, 두 방식에서 대화 기록이 어디에 저장되고 어떻게 다음 요청에 들어가는지 비교해 달라고 합니다. 코드는 고치지 말라고 적습니다.

**성공 기준**

- 🟢 5장은 체인 밖의 `session_store`, 6장은 State 안의 `messages`에 기록이 있다고 짚었다
- 🟢 대화방을 가르는 값이 5장은 `session_id`, 6장은 `thread_id`라고 짚었다
- ⛔ 코드를 고치거나 새로 써 왔다 → 고치지 말라고 다시 적습니다

## 7. 정리

체인, 도구 호출, 구조화된 출력, 메모리를 이어 보았습니다.

| 장 | 무엇을 연결했나 | 결과가 어디에 나오나 |
| --- | --- | --- |
| 2. LCEL과 Runnable | 프롬프트·모델·파서를 파이프 연산자로, 여러 체인을 dict로 | 답변 텍스트, pros·cons 두 목록 |
| 3. 도구 호출 | 도구 목록을 모델에(`bind_tools`), 도구 이름을 실제 도구에(`tool_map`) | 호출 정보 줄과 도구 결과를 쓴 최종 답 |
| 4. 구조화된 출력 | 스키마를 파서에, 또는 모델에 직접 | 제목·장르·평점·요약·추천 다섯 줄 |
| 5. 세션 메모리 | 체인과 세션 기록 저장소를 | 이름을 기억하는 세션 A, 모르는 세션 B |
| 6. 상태 안에 담는 메모리 | 그래프와 `MemorySaver`를 | 스레드별 대화와 State 스냅샷 |

모델은 도구를 실행하지 않고, 이전 대화를 기억하지도 않습니다. 실행과 기억은 코드가 맡고, LangChain과 LangGraph는 그 일을 정해진 모양으로 묶어 줍니다.

보기를 골라 실행하고, 결과를 성공 기준(🟢/⛔)과 비교한 뒤 정답 보기의 해설로 확인했습니다.